# 06 - Integracion con Trakt API

Notebook para conectar el proyecto con Trakt y extraer dos fuentes de datos del usuario:
- peliculas valoradas;
- peliculas vistas.

El objetivo aqui es preparar los datos de forma local y segura. No se conectan todavia estos datos al recomendador.

## 1. Configuracion

Cargamos las variables de entorno desde `.env`, comprobamos que existen las credenciales de Trakt y definimos las rutas del proyecto con `pathlib`.

Por seguridad, nunca se muestra `TRAKT_CLIENT_SECRET` por pantalla.

In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import json
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

base_dir = Path.cwd().resolve()
if not (base_dir / 'data').exists():
    for parent in base_dir.parents:
        if (parent / 'data').exists():
            base_dir = parent
            break

processed_dir = base_dir / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

env_path = base_dir / '.env'
load_dotenv(env_path)

TRAKT_CLIENT_ID = os.getenv('TRAKT_CLIENT_ID', '').strip()
TRAKT_CLIENT_SECRET = os.getenv('TRAKT_CLIENT_SECRET', '').strip()
TRAKT_SCOPE = os.getenv('TRAKT_SCOPE', 'public').strip()

print('Base dir:', base_dir)
print('Env loaded:', env_path.exists())
print('Client ID loaded:', bool(TRAKT_CLIENT_ID))
print('Client secret loaded:', bool(TRAKT_CLIENT_SECRET))
print('Scope:', TRAKT_SCOPE if TRAKT_SCOPE else '(empty)')

if not TRAKT_CLIENT_ID or not TRAKT_CLIENT_SECRET:
    raise ValueError('Faltan TRAKT_CLIENT_ID o TRAKT_CLIENT_SECRET en el archivo .env.')

TRAKT_API_BASE = 'https://api.trakt.tv'
token_path = processed_dir / 'trakt_token.json'
ratings_csv_path = processed_dir / 'trakt_ratings.csv'
watched_csv_path = processed_dir / 'trakt_watched.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 100)

Base dir: C:\Users\alexo\Desktop\TFG
Env loaded: True
Client ID loaded: True
Client secret loaded: True
Scope: public


## 2. Autenticacion por dispositivo

Usamos el flujo de device code de Trakt:
1. pedimos un `user_code` y una `verification_url`;
2. el usuario autoriza la app en el navegador;
3. hacemos polling hasta obtener el `access_token`;
4. guardamos el token localmente en `data/processed/trakt_token.json`.

Si ya existe un token valido, el notebook lo reutiliza para no pedir autorizacion otra vez.

In [2]:
session = requests.Session()


def build_public_headers():
    return {
        'trakt-api-version': '2',
        'trakt-api-key': TRAKT_CLIENT_ID,
        'Content-Type': 'application/json',
        'Accept': 'application/json',
    }


def build_auth_headers(access_token):
    headers = build_public_headers().copy()
    headers['Authorization'] = f'Bearer {access_token}'
    return headers


def load_saved_token():
    if not token_path.exists() or token_path.stat().st_size == 0:
        return None
    try:
        with token_path.open('r', encoding='utf-8') as handle:
            token_data = json.load(handle)
    except Exception:
        return None
    return token_data if isinstance(token_data, dict) else None


def save_token(token_data):
    token_copy = dict(token_data)
    token_copy['created_at'] = datetime.now(timezone.utc).isoformat()
    expires_in = int(token_copy.get('expires_in', 0) or 0)
    if expires_in > 0:
        token_copy['expires_at'] = (datetime.now(timezone.utc) + timedelta(seconds=expires_in)).isoformat()
    with token_path.open('w', encoding='utf-8') as handle:
        json.dump(token_copy, handle, indent=2, ensure_ascii=False)
    return token_copy


def token_is_valid(token_data):
    if not token_data or 'access_token' not in token_data:
        return False
    expires_at = token_data.get('expires_at')
    if not expires_at:
        return True
    try:
        expiry = datetime.fromisoformat(expires_at)
    except Exception:
        return False
    return datetime.now(timezone.utc) < expiry


def request_device_code():
    payload = {'client_id': TRAKT_CLIENT_ID}
    if TRAKT_SCOPE:
        payload['scope'] = TRAKT_SCOPE
    response = session.post(
        f'{TRAKT_API_BASE}/oauth/device/code',
        headers=build_public_headers(),
        json=payload,
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def poll_device_token(device_code, interval=5, expires_in=600):
    deadline = time.time() + int(expires_in)
    poll_interval = max(int(interval), 1)
    attempt = 0

    while time.time() < deadline:
        attempt += 1
        print(f"Esperando autorización en Trakt... intento {attempt}", flush=True)

        response = session.post(
            f"{TRAKT_API_BASE}/oauth/device/token",
            headers=build_public_headers(),
            json={
                "code": device_code,
                "client_id": TRAKT_CLIENT_ID,
                "client_secret": TRAKT_CLIENT_SECRET,
            },
            timeout=30,
        )

        print("Status:", response.status_code, flush=True)
        print("Respuesta:", response.text[:300], flush=True)

        if response.status_code == 200:
            return response.json()

        # Trakt puede responder 400 mientras la autorización sigue pendiente.
        try:
            error_payload = response.json()
        except Exception:
            error_payload = {}

        error = error_payload.get("error")
        error_description = error_payload.get("error_description")

        if response.status_code == 400 and not error:
            print("Aún no autorizado o respuesta sin JSON. Esperando...", flush=True)
            time.sleep(poll_interval)
            continue

        if error == "authorization_pending":
            print("Autorización pendiente. Autoriza el código en el navegador.", flush=True)
            time.sleep(poll_interval)
            continue

        if error == "slow_down":
            poll_interval += 5
            print(f"Trakt pide ir más despacio. Nuevo intervalo: {poll_interval}s", flush=True)
            time.sleep(poll_interval)
            continue

        if error in {"access_denied", "expired_token"}:
            raise RuntimeError(f"Trakt device auth failed: {error} - {error_description}")

        if response.status_code in {400, 404}:
            print("Todavía pendiente. Esperando...", flush=True)
            time.sleep(poll_interval)
            continue

        raise RuntimeError(
            f"Unexpected Trakt auth error. "
            f"Status={response.status_code}, body={response.text}"
        )

    raise TimeoutError("Se agotó el tiempo de espera de la autorización por dispositivo.")


def get_access_token():
    saved_token = load_saved_token()
    if token_is_valid(saved_token):
        print('Usando token guardado en caché local.')
        return saved_token

    device_code_data = request_device_code()
    verification_url = device_code_data.get('verification_url') or device_code_data.get('verification_uri')
    user_code = device_code_data.get('user_code')
    device_code = device_code_data.get('device_code')
    interval = device_code_data.get('interval', 5)
    expires_in = device_code_data.get('expires_in', 600)

    print('Abre esta URL en tu navegador:', verification_url)
    print('Introduce este codigo:', user_code)

    token_data = poll_device_token(device_code, interval=interval, expires_in=expires_in)
    saved = save_token(token_data)
    print('Token guardado en:', token_path)
    return saved


token_data = get_access_token()
access_token = token_data['access_token']
auth_headers = build_auth_headers(access_token)

print('Token cargado correctamente.')
print('Token type:', token_data.get('token_type'))
print('Scope concedido:', token_data.get('scope'))

Usando token guardado en caché local.
Token cargado correctamente.
Token type: bearer
Scope concedido: public


## 3. Comprobacion de conexion

Antes de descargar datos de sincronizacion, comprobamos que la autenticacion funciona con `/users/settings`.

In [3]:
def trakt_get(path):
    response = session.get(f'{TRAKT_API_BASE}{path}', headers=auth_headers, timeout=30)
    response.raise_for_status()
    return response.json()


user_settings = trakt_get('/users/settings')
display(pd.DataFrame([user_settings]))

,user,account,connections,sharing_text,limits,permissions
0,"{'username': 'alexonazo', 'private': False, 'deleted': False, 'name': 'Alex Bustillo', 'vip': Fa...","{'timezone': 'Europe/Madrid', 'date_format': 'dmy', 'time_24hr': False, 'cover_image': None, 'to...","{'facebook': False, 'twitter': False, 'mastodon': False, 'google': True, 'tumblr': False, 'mediu...","{'watching': 'I'm watching [item]', 'watched': 'I just watched [item]', 'rated': None}","{'list': {'count': 4, 'item_count': 250}, 'watchlist': {'item_count': 250}, 'favorites': {'item_...","{'commenting': True, 'liking': True, 'following': True}"


## 4. Descarga de ratings y vistos

Con la sesion autenticada descargamos dos endpoints:
- `/sync/ratings/movies`;
- `/sync/watched/movies`.

La respuesta se convierte a DataFrames para trabajarla despues en el proyecto.

In [4]:
trakt_ratings_json = trakt_get('/sync/ratings/movies')
trakt_watched_json = trakt_get('/sync/watched/movies')

print('Ratings recibidos:', len(trakt_ratings_json))
print('Watched recibidos:', len(trakt_watched_json))

Ratings recibidos: 147
Watched recibidos: 256


## 5. Normalizacion a DataFrames

Aplanamos ambas respuestas JSON a un formato tabular con las columnas necesarias para futuras analisis.

Campos principales:
- `title`, `year`, `trakt_id`, `slug`, `imdb_id`, `tmdb_id`;
- `user_rating_trakt`, `user_rating_normalized`, `rated_at`;
- `plays`, `last_watched_at`.

Las columnas que no existan en cada endpoint se dejan como valores nulos para mantener un esquema estable.

In [5]:
import numpy as np

def flatten_movie_item(item, source):
    movie = item.get('movie', {}) or {}
    ids = movie.get('ids', {}) or {}
    trakt_rating = item.get('rating')

    normalized_rating = np.nan
    if trakt_rating is not None:
        try:
            normalized_rating = float(trakt_rating) / 2.0
        except Exception:
            normalized_rating = np.nan

    return {
        'title': movie.get('title'),
        'year': movie.get('year'),
        'trakt_id': ids.get('trakt'),
        'slug': ids.get('slug'),
        'imdb_id': ids.get('imdb'),
        'tmdb_id': ids.get('tmdb'),
        'user_rating_trakt': trakt_rating if source == 'ratings' else np.nan,
        'user_rating_normalized': normalized_rating if source == 'ratings' else np.nan,
        'rated_at': item.get('rated_at') if source == 'ratings' else np.nan,
        'plays': item.get('plays') if source == 'watched' else np.nan,
        'last_watched_at': item.get('last_watched_at') if source == 'watched' else np.nan,
    }


trakt_ratings_df = pd.DataFrame([flatten_movie_item(item, 'ratings') for item in trakt_ratings_json])
trakt_watched_df = pd.DataFrame([flatten_movie_item(item, 'watched') for item in trakt_watched_json])

display(trakt_ratings_df.head())
display(trakt_watched_df.head())

,title,year,trakt_id,slug,imdb_id,tmdb_id,user_rating_trakt,user_rating_normalized,rated_at,plays,last_watched_at
0,Sundays,2025,1181529,sundays-2025,tt35513027,1443576,6,3.0,2026-05-01T16:15:07.000Z,NaN,NaN
1,Cinema Paradiso,1988,6342,cinema-paradiso-1988,tt0095765,11216,8,4.0,2026-05-01T16:14:43.000Z,NaN,NaN
2,Dora and the Lost City of Gold,2019,346883,dora-and-the-lost-city-of-gold-2019,tt7547410,499701,6,3.0,2026-03-26T18:49:17.000Z,NaN,NaN
3,The Housemaid,2025,1117195,the-housemaid-2025,tt27543632,1368166,4,2.0,2026-03-26T18:49:01.000Z,NaN,NaN
4,Marty Supreme,2025,1076665,marty-supreme-2025,tt32916440,1317288,10,5.0,2026-03-19T16:23:23.000Z,NaN,NaN


,title,year,trakt_id,slug,imdb_id,tmdb_id,user_rating_trakt,user_rating_normalized,rated_at,plays,last_watched_at
0,Hoppers,2026,1085761,hoppers-2026,tt26443616,1327819,NaN,NaN,NaN,4,2026-03-29T00:54:00.000Z
1,Ted,2012,52936,ted-2012,tt1637725,72105,NaN,NaN,NaN,3,2026-04-26T14:11:00.000Z
2,Coraline,2009,8984,coraline-2009,tt0327597,14836,NaN,NaN,NaN,2,2026-01-30T18:46:00.000Z
3,Whiplash,2014,149047,whiplash-2014,tt2582802,244786,NaN,NaN,NaN,2,2026-04-29T23:24:00.000Z
4,The Bad Guys,2022,469856,the-bad-guys-2022,tt8115900,629542,NaN,NaN,NaN,2,2026-02-08T23:59:00.000Z


## 6. Guardado local

Guardamos los resultados en CSV para poder usarlos despues en el proyecto o en Power BI.

Esta parte guarda los CSV de Trakt; despues se reutilizan para mapearlos con MovieLens y preparar el perfil por IDs.

In [6]:
trakt_ratings_df.to_csv(ratings_csv_path, index=False, encoding='utf-8-sig')
trakt_watched_df.to_csv(watched_csv_path, index=False, encoding='utf-8-sig')

print('Guardado en:', ratings_csv_path)
print('Guardado en:', watched_csv_path)
print('Ratings shape:', trakt_ratings_df.shape)
print('Watched shape:', trakt_watched_df.shape)

Guardado en: C:\Users\alexo\Desktop\TFG\data\processed\trakt_ratings.csv
Guardado en: C:\Users\alexo\Desktop\TFG\data\processed\trakt_watched.csv
Ratings shape: (147, 11)
Watched shape: (256, 11)


## 7. Mapeo de Trakt con MovieLens

Ahora usamos los identificadores compartidos entre ambos mundos para pasar de las peliculas de Trakt a `movieId` de MovieLens.

Primero intentamos el mapeo por `tmdbId` y, si no hay coincidencia, completamos con `imdbId`.

In [7]:
import re

import numpy as np
import pandas as pd

movies_clean_path = base_dir / 'data' / 'processed' / 'movies_clean.csv'
links_path = base_dir / 'data' / 'raw' / 'links.csv'

trakt_ratings_df = pd.read_csv(ratings_csv_path)
trakt_watched_df = pd.read_csv(watched_csv_path)
links_df = pd.read_csv(links_path, dtype={'movieId': 'Int64', 'imdbId': 'string', 'tmdbId': 'string'})
movies_clean_df = pd.read_csv(movies_clean_path)

trakt_ratings_df['user_rating_normalized'] = pd.to_numeric(trakt_ratings_df['user_rating_trakt'], errors='coerce') / 2.0

def normalize_imdb_id(value):
    if pd.isna(value):
        return ''
    text = str(value).strip().lower()
    if not text:
        return ''
    numeric = pd.to_numeric(text, errors='coerce')
    if pd.notna(numeric):
        return str(int(numeric)).zfill(7)
    digits = re.sub(r'\D', '', text)
    return digits.zfill(7) if digits else ''


def normalize_tmdb_id(value):
    if pd.isna(value):
        return ''
    text = str(value).strip()
    if not text:
        return ''
    numeric = pd.to_numeric(text, errors='coerce')
    if pd.notna(numeric):
        return str(int(numeric))
    digits = re.sub(r'\D', '', text)
    return str(int(digits)) if digits else ''


trakt_ratings_df['imdbId_norm'] = trakt_ratings_df['imdb_id'].map(normalize_imdb_id)
trakt_ratings_df['tmdbId_norm'] = trakt_ratings_df['tmdb_id'].map(normalize_tmdb_id)
trakt_watched_df['imdbId_norm'] = trakt_watched_df['imdb_id'].map(normalize_imdb_id)
trakt_watched_df['tmdbId_norm'] = trakt_watched_df['tmdb_id'].map(normalize_tmdb_id)

links_df['movieId'] = pd.to_numeric(links_df['movieId'], errors='coerce').astype('Int64')
links_df['imdbId_norm'] = links_df['imdbId'].map(normalize_imdb_id)
links_df['tmdbId_norm'] = links_df['tmdbId'].map(normalize_tmdb_id)

tmdb_lookup = (
    links_df.dropna(subset=['movieId'])
    .drop_duplicates(subset=['tmdbId_norm'], keep='first')
    .set_index('tmdbId_norm')['movieId']
    .to_dict()
)
imdb_lookup = (
    links_df.dropna(subset=['movieId'])
    .drop_duplicates(subset=['imdbId_norm'], keep='first')
    .set_index('imdbId_norm')['movieId']
    .to_dict()
)


def map_trakt_to_movielens(df):
    mapped = df.copy()
    mapped['movieId_tmdb'] = mapped['tmdbId_norm'].map(tmdb_lookup)
    mapped['movieId_imdb'] = mapped['imdbId_norm'].map(imdb_lookup)
    mapped['movieId'] = mapped['movieId_tmdb'].fillna(mapped['movieId_imdb'])
    mapped['movieId'] = pd.to_numeric(mapped['movieId'], errors='coerce').astype('Int64')
    mapped['match_source'] = np.select(
        [mapped['movieId_tmdb'].notna(), mapped['movieId_imdb'].notna()],
        ['tmdb', 'imdb'],
        default='unmapped',
    )
    return mapped


movies_lookup = movies_clean_df[[
    'movieId',
    'title',
    'title_clean',
    'year',
    'genres',
    'rating_mean',
    'rating_count',
]].copy().rename(columns={
    'title': 'title_ml',
    'title_clean': 'title_clean_ml',
    'year': 'year_ml',
    'genres': 'genres_ml',
    'rating_mean': 'rating_mean_ml',
    'rating_count': 'rating_count_ml',
})

trakt_ratings_mapped = map_trakt_to_movielens(trakt_ratings_df).merge(movies_lookup, on='movieId', how='left')
trakt_watched_mapped = map_trakt_to_movielens(trakt_watched_df).merge(movies_lookup, on='movieId', how='left')

mapping_stats = pd.DataFrame([
    {
        'dataset': 'ratings',
        'total': len(trakt_ratings_mapped),
        'mapped': trakt_ratings_mapped['movieId'].notna().sum(),
        'not_mapped': trakt_ratings_mapped['movieId'].isna().sum(),
    },
    {
        'dataset': 'watched',
        'total': len(trakt_watched_mapped),
        'mapped': trakt_watched_mapped['movieId'].notna().sum(),
        'not_mapped': trakt_watched_mapped['movieId'].isna().sum(),
    },
])

display(mapping_stats)
display(trakt_ratings_mapped.head())
display(trakt_watched_mapped.head())

trakt_ratings_mapped.to_csv(processed_dir / 'trakt_ratings_mapped.csv', index=False, encoding='utf-8-sig')
trakt_watched_mapped.to_csv(processed_dir / 'trakt_watched_mapped.csv', index=False, encoding='utf-8-sig')

print('Guardado en:', processed_dir / 'trakt_ratings_mapped.csv')
print('Guardado en:', processed_dir / 'trakt_watched_mapped.csv')

,dataset,total,mapped,not_mapped
0,ratings,147,123,24
1,watched,256,218,38


,title,year,trakt_id,slug,imdb_id,tmdb_id,user_rating_trakt,user_rating_normalized,rated_at,plays,last_watched_at,imdbId_norm,tmdbId_norm,movieId_tmdb,movieId_imdb,movieId,match_source,title_ml,title_clean_ml,year_ml,genres_ml,rating_mean_ml,rating_count_ml
0,Sundays,2025,1181529,sundays-2025,tt35513027,1443576,6,3.0,2026-05-01T16:15:07.000Z,NaN,NaN,35513027,1443576,NaN,NaN,<NA>,unmapped,NaN,NaN,NaN,NaN,NaN,NaN
1,Cinema Paradiso,1988,6342,cinema-paradiso-1988,tt0095765,11216,8,4.0,2026-05-01T16:14:43.000Z,NaN,NaN,0095765,11216,1172.0,1172.0,1172,tmdb,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Cinema Paradiso (Nuovo cinema Paradiso),1989.0,Drama,4.129985,10982.0
2,Dora and the Lost City of Gold,2019,346883,dora-and-the-lost-city-of-gold-2019,tt7547410,499701,6,3.0,2026-03-26T18:49:17.000Z,NaN,NaN,7547410,499701,203224.0,203224.0,203224,tmdb,Dora and the Lost City of Gold (2019),Dora and the Lost City of Gold,2019.0,Adventure|Children|Comedy,2.919431,211.0
3,The Housemaid,2025,1117195,the-housemaid-2025,tt27543632,1368166,4,2.0,2026-03-26T18:49:01.000Z,NaN,NaN,27543632,1368166,NaN,NaN,<NA>,unmapped,NaN,NaN,NaN,NaN,NaN,NaN
4,Marty Supreme,2025,1076665,marty-supreme-2025,tt32916440,1317288,10,5.0,2026-03-19T16:23:23.000Z,NaN,NaN,32916440,1317288,NaN,NaN,<NA>,unmapped,NaN,NaN,NaN,NaN,NaN,NaN


,title,year,trakt_id,slug,imdb_id,tmdb_id,user_rating_trakt,user_rating_normalized,rated_at,plays,last_watched_at,imdbId_norm,tmdbId_norm,movieId_tmdb,movieId_imdb,movieId,match_source,title_ml,title_clean_ml,year_ml,genres_ml,rating_mean_ml,rating_count_ml
0,Hoppers,2026,1085761,hoppers-2026,tt26443616,1327819,NaN,NaN,NaN,4,2026-03-29T00:54:00.000Z,26443616,1327819,NaN,NaN,<NA>,unmapped,NaN,NaN,NaN,NaN,NaN,NaN
1,Ted,2012,52936,ted-2012,tt1637725,72105,NaN,NaN,NaN,3,2026-04-26T14:11:00.000Z,1637725,72105,95441.0,95441.0,95441,tmdb,Ted (2012),Ted,2012.0,Comedy|Fantasy,3.130762,8439.0
2,Coraline,2009,8984,coraline-2009,tt0327597,14836,NaN,NaN,NaN,2,2026-01-30T18:46:00.000Z,0327597,14836,66097.0,66097.0,66097,tmdb,Coraline (2009),Coraline,2009.0,Animation|Fantasy|Thriller,3.747544,9671.0
3,Whiplash,2014,149047,whiplash-2014,tt2582802,244786,NaN,NaN,NaN,2,2026-04-29T23:24:00.000Z,2582802,244786,112552.0,112552.0,112552,tmdb,Whiplash (2014),Whiplash,2014.0,Drama,4.154747,19477.0
4,The Bad Guys,2022,469856,the-bad-guys-2022,tt8115900,629542,NaN,NaN,NaN,2,2026-02-08T23:59:00.000Z,8115900,629542,272407.0,272407.0,272407,tmdb,The Bad Guys (2022),The Bad Guys,2022.0,Action|Animation|Children|Comedy|Crime,3.292636,258.0


Guardado en: C:\Users\alexo\Desktop\TFG\data\processed\trakt_ratings_mapped.csv
Guardado en: C:\Users\alexo\Desktop\TFG\data\processed\trakt_watched_mapped.csv


## 8. Preparacion del perfil compatible con el recomendador

Para que Trakt pueda alimentar el recomendador explicable, reconstruimos la tabla de peliculas con tags normalizados, la matriz TF-IDF y las variables de calidad y popularidad.

La traduccion de tags no se hace aqui con APIs externas: reutilizamos el caché local ya generado en el proyecto.

In [8]:
import unicodedata

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tags_path = base_dir / 'data' / 'raw' / 'tags.csv'
translation_cache_path = base_dir / 'data' / 'processed' / 'tag_translation_cache.csv'

def normalize_text(text):
    if pd.isna(text):
        return ''
    text = str(text).strip().lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(character for character in text if not unicodedata.combining(character))
    text = re.sub(r'[^a-z0-9\s-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def is_useful_tag(tag):
    cleaned = normalize_text(tag)
    if not cleaned:
        return False
    if not re.search(r'[a-z]', cleaned):
        return False
    if re.fullmatch(r'[\d\s\-/]+', cleaned):
        return False
    letters = len(re.findall(r'[a-z]', cleaned))
    digits = len(re.findall(r'\d', cleaned))
    if digits > letters:
        return False
    if ' ' not in cleaned and len(cleaned) <= 3:
        return False
    return True


def normalize_series(series):
    numeric = pd.to_numeric(series, errors='coerce')
    min_value = numeric.min()
    max_value = numeric.max()
    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(0.5, index=series.index)
    return (numeric - min_value) / (max_value - min_value)


MIN_TAG_FREQUENCY = 2
tags_df = pd.read_csv(tags_path)
tags_clean = tags_df[['movieId', 'tag']].dropna(subset=['tag']).copy()
tags_clean['tag_clean'] = tags_clean['tag'].map(normalize_text)
tags_clean = tags_clean[tags_clean['tag_clean'].map(is_useful_tag)].copy()
tags_clean = tags_clean.drop_duplicates(subset=['movieId', 'tag_clean']).reset_index(drop=True)

tag_frequency = tags_clean['tag_clean'].value_counts()
tags_clean = tags_clean[tags_clean['tag_clean'].map(tag_frequency) >= MIN_TAG_FREQUENCY].copy().reset_index(drop=True)

if translation_cache_path.exists() and translation_cache_path.stat().st_size > 0:
    translation_cache = pd.read_csv(translation_cache_path)
    if {'tag_clean', 'tag_translated'}.issubset(translation_cache.columns):
        translation_cache = translation_cache[['tag_clean', 'tag_translated']].copy()
        translation_cache['tag_clean'] = translation_cache['tag_clean'].map(normalize_text)
        translation_cache['tag_translated'] = translation_cache['tag_translated'].fillna('').map(normalize_text)
        translation_cache = translation_cache[
            (translation_cache['tag_clean'] != '')
            & (translation_cache['tag_translated'] != '')
        ].drop_duplicates(subset=['tag_clean'], keep='last')
    else:
        translation_cache = pd.DataFrame(columns=['tag_clean', 'tag_translated'])
else:
    translation_cache = pd.DataFrame(columns=['tag_clean', 'tag_translated'])

translation_map = dict(zip(translation_cache['tag_clean'], translation_cache['tag_translated']))
tags_clean['tag_en'] = tags_clean['tag_clean'].map(translation_map).fillna(tags_clean['tag_clean'])
tags_clean['tag_en'] = tags_clean['tag_en'].map(normalize_text)
tags_clean = tags_clean[tags_clean['tag_en'].map(is_useful_tag)].copy()
tags_clean = tags_clean.drop_duplicates(subset=['movieId', 'tag_en']).reset_index(drop=True)

tags_grouped_en = (
    tags_clean.groupby('movieId')['tag_en']
    .apply(lambda values: ' '.join(sorted(set(tag for tag in values if tag))))
    .reset_index(name='tags_combined_en')
)

movies_with_tags = movies_clean_df.merge(tags_grouped_en, on='movieId', how='left')
movies_with_tags['tags_combined_en'] = movies_with_tags['tags_combined_en'].fillna('')
movies_with_tags['genres_text'] = movies_with_tags['genres'].fillna('').str.replace('|', ' ', regex=False)
movies_with_tags['content_features'] = (
    movies_with_tags['genres_text'].str.strip() + ' ' + movies_with_tags['tags_combined_en'].str.strip()
).str.strip()

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words='english',
    min_df=2,
    max_df=0.85,
    token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z0-9\-]{2,}\b',
)
content_matrix = tfidf_vectorizer.fit_transform(movies_with_tags['content_features'])
feature_names = tfidf_vectorizer.get_feature_names_out()

movies_advanced = movies_with_tags.copy()
movies_advanced['rating_count'] = pd.to_numeric(movies_advanced['rating_count'], errors='coerce').fillna(0)
movies_advanced['rating_mean'] = pd.to_numeric(movies_advanced['rating_mean'], errors='coerce')
movies_advanced['year'] = pd.to_numeric(movies_advanced['year'], errors='coerce').astype('Int64')

m = 20
C = movies_advanced['rating_mean'].mean()
v = movies_advanced['rating_count']
R = movies_advanced['rating_mean'].fillna(C)

movies_advanced['weighted_rating'] = ((v / (v + m)) * R) + ((m / (v + m)) * C)
movies_advanced['rating_score'] = normalize_series(movies_advanced['weighted_rating'])
movies_advanced['popularity_score'] = normalize_series(np.log1p(movies_advanced['rating_count']))

movieId_to_index = movies_advanced.reset_index().set_index('movieId')['index'].to_dict()

print('Matriz TF-IDF:', content_matrix.shape)
display(movies_advanced[['title', 'year', 'genres', 'tags_combined_en', 'weighted_rating', 'rating_score', 'popularity_score']].head())

Matriz TF-IDF: (87585, 24664)


,title,year,genres,tags_combined_en,weighted_rating,rating_score,popularity_score
0,Toy Story (1995),1995,Adventure|Animation|Children|Comedy|Fantasy,2009 reissue in stereoscopic 3-d 3 dimensional 55 movies every kid should see--entertainment wee...,3.897179,0.842583,0.965346
1,Jumanji (1995),1995,Adventure|Children|Fantasy,19th century 20th century abandoned factory actor playing multiple roles adapted from book adven...,3.275571,0.660531,0.889962
2,Grumpier Old Men (1995),1995,Comedy|Romance,best friend duringcreditsstinger fishing funniest movies funny grun running jack lemmon midwest ...,3.139243,0.620604,0.821625
3,Waiting to Exhale (1995),1995,Comedy|Drama|Romance,based on novel or book characters chick flick divorce interracial relationship revenge single mo...,2.846462,0.534856,0.687923
4,Father of the Bride Part II (1995),1995,Comedy,4th wall aging baby comedy confidence contraception daughter diane keaton family fantasy father ...,3.059519,0.597255,0.821757


## 9. Recomendador por perfil de Trakt

Con los `movieId` ya alineados, construimos un perfil positivo y uno negativo a partir de las valoraciones reales de Trakt.

La nueva variante no rompe el recomendador anterior por titulos: simplemente añade compatibilidad para trabajar directamente con IDs.

In [9]:
def _movie_index_from_id(movie_id):
    if pd.isna(movie_id):
        return None
    try:
        movie_id_int = int(movie_id)
    except Exception:
        return None
    movie_index = movieId_to_index.get(movie_id_int)
    if movie_index is None:
        return None
    return int(movie_index)


def _weighted_profile_vector(movie_indices, weights):
    if not movie_indices:
        return None

    weights = np.asarray(weights, dtype=float)
    if weights.sum() == 0:
        weights = np.ones_like(weights)

    weighted_vectors = content_matrix[movie_indices].multiply(weights[:, None])
    profile = weighted_vectors.sum(axis=0) / weights.sum()
    return np.asarray(profile)


def _split_genres(genres_value):
    if pd.isna(genres_value):
        return []
    return [genre.strip() for genre in str(genres_value).split('|') if genre.strip()]


def _top_profile_terms(profile_vector, top_n=15):
    if profile_vector is None:
        return pd.DataFrame(columns=['feature', 'weight'])

    weights = np.asarray(profile_vector).ravel()
    positive_indices = np.where(weights > 0)[0]
    if len(positive_indices) == 0:
        return pd.DataFrame(columns=['feature', 'weight'])

    top_indices = positive_indices[np.argsort(weights[positive_indices])[::-1]][:top_n]
    return pd.DataFrame({'feature': feature_names[top_indices], 'weight': weights[top_indices]})


def _weighted_genre_summary(profile_movies, top_n=15):
    rows = []
    for _, row in profile_movies.iterrows():
        for genre in _split_genres(row['genres']):
            rows.append({'genre': genre, 'weight': row['user_rating']})
    if not rows:
        return pd.DataFrame(columns=['genre', 'weight'])
    return (
        pd.DataFrame(rows)
        .groupby('genre', as_index=False)['weight'].sum()
        .sort_values('weight', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


MODE_WEIGHTS = {
    'balanced': {'similarity': 0.70, 'negative': -0.20, 'rating': 0.08, 'popularity': 0.02, 'middle_popularity': 0.00, 'low_popularity': 0.00},
    'popular': {'similarity': 0.40, 'negative': -0.15, 'rating': 0.10, 'popularity': 0.35, 'middle_popularity': 0.00, 'low_popularity': 0.00},
    'under_the_radar': {'similarity': 0.50, 'negative': -0.20, 'rating': 0.25, 'popularity': 0.00, 'middle_popularity': 0.15, 'low_popularity': 0.00},
    'obscure': {'similarity': 0.60, 'negative': -0.20, 'rating': 0.20, 'popularity': 0.00, 'middle_popularity': 0.00, 'low_popularity': 0.10},
    'quality': {'similarity': 0.40, 'negative': -0.15, 'rating': 0.35, 'popularity': 0.10, 'middle_popularity': 0.00, 'low_popularity': 0.00},
}


def _normalize_genre_list(genres):
    if genres is None:
        return []
    if isinstance(genres, str):
        genres = [genres]
    return [str(genre).strip().lower() for genre in genres if str(genre).strip()]


def _movie_genre_set(genres_value):
    return {genre.lower() for genre in _split_genres(genres_value)}


def _apply_filters(df, min_ratings=20, min_year=None, max_year=None, include_genres=None, exclude_genres=None):
    result = df.copy()
    if min_ratings is not None:
        result = result[result['rating_count'] >= min_ratings]
    if min_year is not None:
        result = result[result['year'].fillna(-1) >= min_year]
    if max_year is not None:
        result = result[result['year'].fillna(10**9) <= max_year]

    include_genres = _normalize_genre_list(include_genres)
    exclude_genres = _normalize_genre_list(exclude_genres)
    if include_genres:
        result = result[result['genres'].apply(lambda value: set(include_genres).issubset(_movie_genre_set(value)))]
    if exclude_genres:
        result = result[~result['genres'].apply(lambda value: bool(_movie_genre_set(value) & set(exclude_genres)))]
    return result


def _popularity_label(score):
    if score >= 0.75:
        return 'popularidad alta'
    if score >= 0.40:
        return 'popularidad media'
    return 'popularidad baja'


EXPLANATION_STOP_TERMS = {
    'nominee', 'nominated', 'nomination', 'award', 'awards', 'oscar', 'oscars', 'afi',
    'reference', 'references', 'relationship', 'relationships', 'male', 'female',
    'man', 'woman', 'boy', 'girl', 'title', 'character', 'characters', 'soundtrack',
    'score', 'actor', 'actress', 'director', 'starring', 'performance', 'performances',
    'best', 'great', 'good', 'bad', 'movie', 'film', 'films', 'cinema', 'classic',
    'favorite', 'favourite', 'seen', 'watched', 'dvd', 'blu-ray', 'netflix',
    '100', '20th', '21st', 'century', 'stars',
}


def is_explainable_term(term):
    cleaned = normalize_text(term)
    if not cleaned:
        return False
    if cleaned in EXPLANATION_STOP_TERMS:
        return False
    if len(cleaned) <= 3:
        return False
    if re.fullmatch(r'[\d\s\-/]+', cleaned):
        return False
    if not re.search(r'[a-z]', cleaned):
        return False
    return True


def get_explainable_terms_from_text(text):
    tokens = [token for token in normalize_text(text).split() if token]
    terms = []
    for token in tokens:
        if is_explainable_term(token) and token not in terms:
            terms.append(token)
    return terms


def _top_movie_terms(movie_index, top_profile_terms, max_terms=3):
    row = content_matrix[movie_index]
    movie_terms = [feature_names[idx] for idx in row.nonzero()[1]]
    movie_terms = [term for term in movie_terms if is_explainable_term(term)]
    movie_terms.extend(get_explainable_terms_from_text(movies_advanced.loc[movie_index, 'tags_combined_en']))
    movie_terms = list(dict.fromkeys(movie_terms))
    profile_terms = [term for term in top_profile_terms if is_explainable_term(term)]
    matches = []
    movie_terms_set = set(movie_terms)
    for term in profile_terms:
        if term in movie_terms_set and term not in matches:
            matches.append(term)
        if len(matches) >= max_terms:
            break
    return matches[:max_terms]


def _top_genre_matches(movie_genres, top_profile_genres, max_terms=3):
    movie_genre_set = {normalize_text(genre) for genre in _split_genres(movie_genres) if normalize_text(genre)}
    profile_genres = [normalize_text(genre) for genre in (top_profile_genres or []) if normalize_text(genre)]
    matches = []
    for genre in profile_genres:
        if genre in movie_genre_set and genre not in matches:
            matches.append(genre)
        if len(matches) >= max_terms:
            break
    return matches[:max_terms]


def _quality_phrase(row):
    if row['rating_score'] >= 0.80:
        return 'una valoración ponderada muy alta'
    if row['rating_score'] >= 0.65:
        return 'una valoración ponderada alta'
    return 'una valoración ponderada razonable'


def _popularity_phrase(row):
    if row['popularity_score'] >= 0.75:
        return 'muchas valoraciones'
    if row['popularity_score'] >= 0.40:
        return 'una popularidad media'
    return 'pocas valoraciones'


def _build_explanation(row, top_profile_terms, top_profile_genres=None, mode='balanced'):
    genre_matches = _top_genre_matches(row['genres'], top_profile_genres, max_terms=3)
    tag_matches = _top_movie_terms(int(row['movie_index']), top_profile_terms, max_terms=3)

    parts = []
    if genre_matches:
        genre_text = ', '.join([genre.title() for genre in genre_matches])
        parts.append(f'sus géneros {genre_text}')
    if tag_matches:
        tag_text = ', '.join(tag_matches[:3])
        parts.append(f'temas como {tag_text}')

    if parts:
        if len(parts) == 1:
            reason = f'Coincide con tu perfil por {parts[0]}'
        else:
            reason = f'Coincide con tu perfil por {parts[0]} y por {parts[1]}'
    else:
        genre_list = [genre.title() for genre in _split_genres(row['genres'])[:3]]
        if genre_list:
            reason = 'Coincide con tu perfil por sus géneros ' + ', '.join(genre_list)
        else:
            reason = 'Coincide con tu perfil de contenido'

    quality = _quality_phrase(row)
    popularity = _popularity_phrase(row)

    fallback_parts = [reason]
    fallback_parts.append(f'tiene {quality}')
    fallback_parts.append(f'{popularity}')

    if mode == 'popular':
        fallback_parts.append('y el modo popular prioriza películas con más presencia y visibilidad')
    elif mode == 'under_the_radar':
        fallback_parts.append('y el modo under_the_radar busca equilibrio entre calidad y descubrimiento')
    elif mode == 'obscure':
        fallback_parts.append('y el modo obscure permite propuestas menos conocidas pero afines a tu perfil')
    elif mode == 'quality':
        fallback_parts.append('y el modo quality da más peso a la valoración ponderada')

    if row['negative_similarity_score'] > 0.15:
        fallback_parts.append('aunque comparte algunos rasgos con películas que valoraste peor')

    explanation = '. '.join(fallback_parts[:3])
    if len(fallback_parts) > 3:
        explanation = '. '.join(fallback_parts[:4])
    return explanation + '.'


def build_user_profiles_by_ids(user_ratings_by_id, watched_movie_ids=None):
    found_rows = []
    not_found = []

    watched_movie_ids = set(int(movie_id) for movie_id in watched_movie_ids or [])

    for movie_id, user_rating in user_ratings_by_id.items():
        movie_index = _movie_index_from_id(movie_id)
        if movie_index is None:
            not_found.append({'movieId': movie_id, 'user_rating': user_rating})
            continue

        movie = movies_advanced.loc[movie_index]
        found_rows.append({
            'movieId': int(movie['movieId']),
            'movie_index': movie_index,
            'title': movie['title'],
            'year': movie['year'],
            'genres': movie['genres'],
            'user_rating': float(user_rating),
            'rating_count': movie['rating_count'],
            'weighted_rating': movie['weighted_rating'],
            'watched': int(movie['movieId']) in watched_movie_ids,
        })

    found_df = pd.DataFrame(found_rows)
    not_found_df = pd.DataFrame(not_found)

    if found_df.empty:
        raise ValueError('No se encontro ninguna pelicula del perfil indicado.')

    positive_df = found_df[found_df['user_rating'] >= 4].copy()
    negative_df = found_df[found_df['user_rating'] <= 2.5].copy()

    if positive_df.empty:
        raise ValueError('El perfil necesita al menos una pelicula positiva con rating >= 4.')

    positive_profile_vector = _weighted_profile_vector(
        positive_df['movie_index'].tolist(),
        positive_df['user_rating'].tolist(),
    )

    negative_profile_vector = None
    if not negative_df.empty:
        negative_weights = (5.5 - negative_df['user_rating']).tolist()
        negative_profile_vector = _weighted_profile_vector(negative_df['movie_index'].tolist(), negative_weights)

    return {
        'positive_profile_vector': positive_profile_vector,
        'negative_profile_vector': negative_profile_vector,
        'found_movies': found_df,
        'not_found_movies': not_found_df,
        'positive_movies': positive_df,
        'negative_movies': negative_df,
    }


def explain_user_profile_by_ids(user_ratings_by_id, watched_movie_ids=None, top_n=15):
    profiles = build_user_profiles_by_ids(user_ratings_by_id, watched_movie_ids=watched_movie_ids)
    positive_movies = profiles['positive_movies']
    negative_movies = profiles['negative_movies']

    year_summary = positive_movies['year'].dropna()
    popularity_summary = positive_movies[['rating_count', 'weighted_rating']].mean(numeric_only=True)

    return {
        'found_movies': profiles['found_movies'],
        'not_found_movies': profiles['not_found_movies'],
        'top_positive_genres': _weighted_genre_summary(positive_movies, top_n=top_n),
        'top_positive_features': _top_profile_terms(profiles['positive_profile_vector'], top_n=top_n),
        'top_negative_features': _top_profile_terms(profiles['negative_profile_vector'], top_n=top_n),
        'positive_year_summary': pd.Series({
            'year_mean': year_summary.mean() if not year_summary.empty else np.nan,
            'year_min': year_summary.min() if not year_summary.empty else np.nan,
            'year_max': year_summary.max() if not year_summary.empty else np.nan,
        }),
        'positive_popularity_summary': pd.Series({
            'rating_count_mean': popularity_summary.get('rating_count', np.nan),
            'weighted_rating_mean': popularity_summary.get('weighted_rating', np.nan),
        }),
    }


def recommend_for_user_profile_by_ids(
    user_ratings_by_id,
    watched_movie_ids=None,
    n=20,
    min_ratings=20,
    min_year=None,
    max_year=None,
    include_genres=None,
    exclude_genres=None,
    mode='balanced',
):
    if mode not in MODE_WEIGHTS:
        raise ValueError(f'Modo no valido. Usa uno de: {list(MODE_WEIGHTS)}')

    profiles = build_user_profiles_by_ids(user_ratings_by_id, watched_movie_ids=watched_movie_ids)
    positive_vector = profiles['positive_profile_vector']
    negative_vector = profiles['negative_profile_vector']

    user_similarity = cosine_similarity(positive_vector, content_matrix).flatten()
    if negative_vector is None:
        negative_similarity = np.zeros(len(movies_advanced))
    else:
        negative_similarity = cosine_similarity(negative_vector, content_matrix).flatten()

    candidates = movies_advanced.copy()
    candidates['movie_index'] = candidates.index
    candidates['user_similarity_score'] = user_similarity
    candidates['negative_similarity_score'] = negative_similarity

    excluded_movie_ids = set(int(movie_id) for movie_id in user_ratings_by_id.keys())
    excluded_movie_ids.update(int(movie_id) for movie_id in watched_movie_ids or [])
    candidates = candidates[~candidates['movieId'].astype('Int64').isin(excluded_movie_ids)]
    candidates = _apply_filters(
        candidates,
        min_ratings=min_ratings,
        min_year=min_year,
        max_year=max_year,
        include_genres=include_genres,
        exclude_genres=exclude_genres,
    )

    if candidates.empty:
        return pd.DataFrame(columns=[
            'title', 'year', 'genres', 'tags_combined_en', 'rating_mean', 'rating_count', 'weighted_rating',
            'rating_score', 'popularity_score', 'user_similarity_score', 'negative_similarity_score',
            'final_score', 'recommendation_mode', 'explanation'
        ])

    weights = MODE_WEIGHTS[mode]
    candidates = candidates.copy()
    candidates['middle_popularity_score'] = 1 - (candidates['popularity_score'] - 0.50).abs() / 0.50
    candidates['middle_popularity_score'] = candidates['middle_popularity_score'].clip(lower=0, upper=1)
    candidates['low_popularity_score'] = 1 - candidates['popularity_score']
    candidates['final_score'] = (
        weights['similarity'] * candidates['user_similarity_score']
        + weights['negative'] * candidates['negative_similarity_score']
        + weights['rating'] * candidates['rating_score']
        + weights['popularity'] * candidates['popularity_score']
        + weights['middle_popularity'] * candidates['middle_popularity_score']
        + weights['low_popularity'] * candidates['low_popularity_score']
    )

    profile_details = explain_user_profile_by_ids(
        user_ratings_by_id,
        watched_movie_ids=watched_movie_ids,
        top_n=30,
    )
    top_profile_terms = profile_details['top_positive_features']['feature'].tolist()
    top_profile_genres = profile_details['top_positive_genres']['genre'].tolist()

    candidates['recommendation_mode'] = mode
    candidates['explanation'] = candidates.apply(
        lambda row: _build_explanation(row, top_profile_terms, top_profile_genres=top_profile_genres, mode=mode),
        axis=1,
    )

    output_columns = [
        'title',
        'year',
        'genres',
        'tags_combined_en',
        'rating_mean',
        'rating_count',
        'weighted_rating',
        'rating_score',
        'popularity_score',
        'user_similarity_score',
        'negative_similarity_score',
        'final_score',
        'recommendation_mode',
        'explanation',
    ]

    return candidates.sort_values('final_score', ascending=False).head(n)[output_columns].reset_index(drop=True)


def recommend_for_trakt_profile(
    user_ratings_by_id,
    watched_movie_ids=None,
    n=20,
    min_ratings=20,
    min_year=None,
    max_year=None,
    include_genres=None,
    exclude_genres=None,
    mode='balanced',
):
    return recommend_for_user_profile_by_ids(
        user_ratings_by_id,
        watched_movie_ids=watched_movie_ids,
        n=n,
        min_ratings=min_ratings,
        min_year=min_year,
        max_year=max_year,
        include_genres=include_genres,
        exclude_genres=exclude_genres,
        mode=mode,
    )


print('Perfil por IDs listo.')
print('Peliculas con features:', movies_advanced.shape[0])

Perfil por IDs listo.
Peliculas con features: 87585


## 10. Ejemplos con datos reales de Trakt

En este bloque usamos directamente las valoraciones reales del usuario en Trakt para construir el perfil y comparar distintos modos de recomendacion.

In [10]:
user_ratings_by_id = (
    trakt_ratings_mapped.dropna(subset=['movieId'])
    .sort_values('rated_at')
    .drop_duplicates(subset=['movieId'], keep='last')
    .assign(movieId=lambda frame: frame['movieId'].astype(int))
    .set_index('movieId')['user_rating_normalized']
    .to_dict()
)

watched_movie_ids = set(
    trakt_watched_mapped.dropna(subset=['movieId'])['movieId'].astype(int).tolist()
)

print('Ratings mapeados en el perfil:', len(user_ratings_by_id))
print('Peliculas vistas mapeadas:', len(watched_movie_ids))
display(pd.DataFrame(list(user_ratings_by_id.items()), columns=['movieId', 'user_rating_normalized']).head())


Ratings mapeados en el perfil: 123
Peliculas vistas mapeadas: 218


,movieId,user_rating_normalized
0,3814,3.5
1,1701,2.5
2,185029,3.5
3,63082,3.5
4,2706,3.0


## 11. Comparacion de modos

Primero resumimos el perfil detectado y despues generamos recomendaciones con varios modos para ver como cambia la lista final.

In [11]:
profile_explanation = explain_user_profile_by_ids(user_ratings_by_id, watched_movie_ids=watched_movie_ids, top_n=12)

display(profile_explanation['found_movies'])
display(profile_explanation['not_found_movies'])
display(profile_explanation['top_positive_genres'])
display(profile_explanation['top_positive_features'])
display(profile_explanation['top_negative_features'])
display(profile_explanation['positive_year_summary'].to_frame().T)
display(profile_explanation['positive_popularity_summary'].to_frame().T)

balanced = recommend_for_trakt_profile(user_ratings_by_id, watched_movie_ids=watched_movie_ids, n=10, mode='balanced')
popular = recommend_for_trakt_profile(user_ratings_by_id, watched_movie_ids=watched_movie_ids, n=10, mode='popular')
under_the_radar = recommend_for_trakt_profile(user_ratings_by_id, watched_movie_ids=watched_movie_ids, n=10, mode='under_the_radar')
obscure = recommend_for_trakt_profile(user_ratings_by_id, watched_movie_ids=watched_movie_ids, n=10, mode='obscure')
quality = recommend_for_trakt_profile(user_ratings_by_id, watched_movie_ids=watched_movie_ids, n=10, mode='quality')

display(balanced)
display(popular)
display(under_the_radar)
display(obscure)
display(quality)

mode_results = []
for mode, recs in [
    ('balanced', balanced),
    ('popular', popular),
    ('under_the_radar', under_the_radar),
    ('obscure', obscure),
    ('quality', quality),
]:
    compact = recs[['title', 'year', 'final_score']].copy()
    compact['mode'] = mode
    mode_results.append(compact)

mode_comparison = pd.concat(mode_results, ignore_index=True)
display(mode_comparison[['mode', 'title', 'year', 'final_score']])


,movieId,movie_index,title,year,genres,user_rating,rating_count,weighted_rating,watched
0,3814,3714,Love and Death (1975),1975,Comedy,3.5,2136,3.938823,True
1,1701,1637,Deconstructing Harry (1997),1997,Comedy|Drama,2.5,3369,3.468752,True
2,185029,52952,A Quiet Place (2018),2018,Drama|Horror|Thriller,3.5,6304,3.694909,True
3,63082,12759,Slumdog Millionaire (2008),2008,Crime|Drama|Romance,3.5,25745,3.821506,True
4,2706,2614,American Pie (1999),1999,Comedy|Romance,3.0,29519,3.275825,True
...,...,...,...,...,...,...,...,...,...
118,106100,20530,Dallas Buyers Club (2013),2013,Drama,3.5,10511,3.902156,True
119,2572,2481,10 Things I Hate About You (1999),1999,Comedy|Romance,3.5,16876,3.555877,True
120,6016,5905,City of God (Cidade de Deus) (2002),2002,Action|Adventure|Crime|Drama|Thriller,4.0,25688,4.177750,True
121,203224,61449,Dora and the Lost City of Gold (2019),2019,Adventure|Children|Comedy,3.0,211,2.926847,True


""


,genre,weight
0,Drama,114.5
1,Comedy,87.0
2,Thriller,49.5
3,Sci-Fi,46.0
4,Crime,41.0
5,Adventure,33.5
6,Fantasy,33.0
7,Romance,28.5
8,Action,28.0
9,Children,25.5


,feature,weight
0,oscar,0.082361
1,ending,0.066616
2,best,0.058261
3,great,0.056355
4,nominee,0.050086
5,reference,0.047670
6,man,0.047598
7,relationship,0.045789
8,car,0.045237
9,soundtrack,0.042217


,feature,weight
0,romance,0.062244
1,drama,0.061521
2,ending,0.050228
3,plot,0.042934
4,character,0.042766
5,thriller,0.039219
6,crime,0.039100
7,oscar,0.039089
8,death,0.038547
9,book,0.037467


...

,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,"Shawshank Redemption, The (1994)",1994,Crime|Drama,100 greatest movies 20th century 20th century literature on screen 5 stars abuse abuse of power ...,4.404614,102929,4.404342,0.991118,1.000000,0.445944,0.314574,0.580303,popular,"Coincide con tu perfil por sus géneros Drama, Crime y por temas como ending, nudity, shot. tiene..."
1,Schindler's List (1993),1993,Drama|War,afi 100 afi 100 cheers afi 100 heroes villains afi 100 movie quotes afi100 altruism amazing phot...,4.236990,73849,4.236657,0.942007,0.971234,0.397414,0.243299,0.556603,popular,"Coincide con tu perfil por sus géneros Drama, War y por temas como nudity, story, music. tiene u..."
2,Fight Club (1999),1999,Action|Crime|Drama|Thriller,20th century fox 20th century studios 5 stars abandoned house absolute favorite acid acting acti...,4.228780,77332,4.228463,0.939608,0.975227,0.427883,0.342162,0.555119,popular,"Coincide con tu perfil por sus géneros Drama, Thriller, Crime y por temas como ending, nudity, s..."
3,Terminator 2: Judgment Day (1991),1991,Action|Sci-Fi,10 year old 20th century 21st century 70mm acquaintance action action girl action hero action he...,3.962045,68383,3.961765,0.861499,0.964571,0.469830,0.387507,0.553556,popular,"Coincide con tu perfil por sus géneros Sci-Fi, Action y por temas como ending, nudity, shot. tie..."
4,One Flew Over the Cuckoo's Nest (1975),1975,Drama,acting adapted from book afi 100 afi 100 cheers afi 100 heroes villains allegory angry anjelica ...,4.204229,44592,4.203692,0.932353,0.927527,0.402046,0.246861,0.541659,popular,"Coincide con tu perfil por sus géneros Drama y por temas como ending, nudity, acting. tiene una ..."
5,Back to the Future (1985),1985,Adventure|Comedy|Sci-Fi,1980s film 20th century 55 movies every kid should see--entertainment weekly 555 phone number 70...,3.969149,61879,3.968838,0.863570,0.955912,0.427165,0.335499,0.541468,popular,"Coincide con tu perfil por sus géneros Comedy, Sci-Fi, Adventure y por temas como ending, shot, ..."
6,"Terminator, The (1984)",1984,Action|Sci-Fi|Thriller,1980s film 20th century 21st century 555 phone number action action girl action hero adventure a...,3.902092,47131,3.901712,0.843911,0.932325,0.446058,0.352352,0.536275,popular,"Coincide con tu perfil por sus géneros Thriller, Sci-Fi, Action y por temas como nudity, shot, d..."
7,Saving Private Ryan (1998),1998,Action|Drama|War,20th century act of god action adam goldberg afi 100 cheers afi 100 thrills airplane almost favo...,4.048742,58368,4.048385,0.886867,0.950851,0.403171,0.313384,0.535746,popular,"Coincide con tu perfil por sus géneros Drama, Action, War y por temas como shot, death, story. t..."
8,"Usual Suspects, The (1995)",1995,Crime|Mystery|Thriller,acting action ambiguous ending anti-hero atmosphere bad acting bd-video benicio del toro boring ...,4.265070,67750,4.264698,0.950220,0.963766,0.349693,0.268495,0.531943,popular,"Coincide con tu perfil por sus géneros Thriller, Crime, Mystery y por temas como ending, story, ..."
9,Braveheart (1995),1995,Action|Drama|War,acting action action packed actor-director adult adventure afi 100 cheers afi 100 thrills allego...,3.988630,69482,3.988347,0.869284,0.965953,0.350692,0.240428,0.529224,popular,"Coincide con tu perfil por sus géneros Drama, Action, War y por temas como ending, nudity, story..."


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,Blindspotting (2018),2018,Comedy|Drama,11 year old 12 year old 26 year old abandoned house african american almost hit by a truck apolo...,3.752599,481,3.722758,0.791500,0.535267,0.345468,0.231673,0.463694,under_the_radar,"Coincide con tu perfil por sus géneros Drama, Comedy y por temas como shot, music, acting. tiene..."
1,"Perfect Crime, The (Crimen Ferpecto) (Ferpect Crime) (2004)",2004,Comedy|Crime|Thriller,8 year old 8 year old girl abuse of power abusive girlfriend accident accidental death accidenta...,3.689024,246,3.637600,0.766560,0.477342,0.367742,0.285727,0.461568,under_the_radar,"Coincide con tu perfil por sus géneros Comedy, Thriller, Crime y por temas como ending, nudity, ..."
2,Pixote (1981),1981,Drama,10 year old abortion abuse of power accidental killing american abroad american man anal rape an...,3.897321,224,3.824187,0.821206,0.469259,0.303919,0.242611,0.449517,under_the_radar,"Coincide con tu perfil por sus géneros Drama y por temas como nudity, shot, death. tiene una val..."
3,Roma (2018),2018,Drama,abandonment absorbing absurdism academic acting adultery airplane alfonso cuaron ambiance artsy ...,3.778936,1918,3.770950,0.805614,0.654972,0.391160,0.266051,0.447282,under_the_radar,"Coincide con tu perfil por sus géneros Drama y por temas como nudity, shot, death. tiene una val..."
4,"Now, Voyager (1942)",1942,Drama|Romance,acting adapted from book adultery afi 100 movie quotes architect artist aunt niece relationship ...,3.948074,597,3.917507,0.848537,0.553951,0.284142,0.222668,0.443486,under_the_radar,"Coincide con tu perfil por sus géneros Drama, Romance y por temas como death, music, acting. tie..."
5,Mishima: A Life in Four Chapters (1985),1985,Drama,actor actress adoration aesthetics air raid air raid shelter air raid warden airplane alarm amer...,3.944853,272,3.880485,0.837694,0.486013,0.264569,0.221172,0.443278,under_the_radar,"Coincide con tu perfil por sus géneros Drama y por temas como nudity, death, story. tiene una va..."
6,The Diary of Anne Frank (1959),1959,Drama|War,adapted from book afi 100 cheers bad adaptation based on a true story biography diary family gir...,3.686497,748,3.668752,0.775683,0.573458,0.321527,0.204617,0.441724,under_the_radar,"Coincide con tu perfil por sus géneros Drama, War y por temas como story. tiene una valoración p..."
7,Vagabond (Sans toit ni loi) (1985),1985,Drama,abandoned building agnes varda agronomist alcoholic alone anonymity apple tree attack aunt niece...,3.871287,303,3.817652,0.819292,0.495332,0.240494,0.164047,0.440860,under_the_radar,"Coincide con tu perfil por sus géneros Drama y por temas como nudity, death, music. tiene una va..."
8,Elevator to the Gallows (a.k.a. Frantic) (Ascenseur pour l'échafaud) (1958),1958,Crime|Drama|Thriller,adulteress adultery alibi arms deal arms dealer atmospheric attempted suicide automobile bare ch...,3.931738,564,3.900003,0.843410,0.549032,0.274116,0.214477,0.440305,under_the_radar,"Coincide con tu perfil por sus géneros Drama, Thriller, Crime y por temas como ending, shot, dea..."
9,"Faces, Places (2017)",2017,Documentary,13 year old 14 year old aerial camera shot animated opening credits art installation art museum ...,3.865285,193,3.784515,0.809587,0.456415,0.275260,0.183608,0.440230,under_the_radar,"Coincide con tu perfil por temas como nudity, shot, death. tiene una valoración ponderada muy al..."


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,"Shawshank Redemption, The (1994)",1994,Crime|Drama,100 greatest movies 20th century 20th century literature on screen 5 stars abuse abuse of power ...,4.404614,102929,4.404342,0.991118,1.000000,0.445944,0.314574,0.402875,obscure,"Coincide con tu perfil por sus géneros Drama, Crime y por temas como ending, nudity, shot. tiene..."
1,One Flew Over the Cuckoo's Nest (1975),1975,Drama,acting adapted from book afi 100 afi 100 cheers afi 100 heroes villains allegory angry anjelica ...,4.204229,44592,4.203692,0.932353,0.927527,0.402046,0.246861,0.385573,obscure,"Coincide con tu perfil por sus géneros Drama y por temas como ending, nudity, acting. tiene una ..."
2,"Graduate, The (1967)",1967,Comedy|Drama|Romance,100 essential female performances 100 greatest movies 20th century 21 year old adapted from book...,4.021081,22319,4.020171,0.878604,0.867563,0.433241,0.333932,0.382123,obscure,"Coincide con tu perfil por sus géneros Drama, Comedy, Romance y por temas como ending, nudity, s..."
3,Schindler's List (1993),1993,Drama|War,afi 100 afi 100 cheers afi 100 heroes villains afi 100 movie quotes afi100 altruism amazing phot...,4.236990,73849,4.236657,0.942007,0.971234,0.397414,0.243299,0.381066,obscure,"Coincide con tu perfil por sus géneros Drama, War y por temas como nudity, story, music. tiene u..."
4,There Will Be Blood (2007),2007,Drama|Western,adaptation adapted from book adopted son adoption alcoholic amazing acting ambition american dre...,4.031561,15494,4.030237,0.881552,0.835941,0.399640,0.257421,0.381016,obscure,"Coincide con tu perfil por sus géneros Drama y por temas como ending, music, acting. tiene una v..."
5,Terminator 2: Judgment Day (1991),1991,Action|Sci-Fi,10 year old 20th century 21st century 70mm acquaintance action action girl action hero action he...,3.962045,68383,3.961765,0.861499,0.964571,0.469830,0.387507,0.380239,obscure,"Coincide con tu perfil por sus géneros Sci-Fi, Action y por temas como ending, nudity, shot. tie..."
6,Lawrence of Arabia (1962),1962,Adventure|Drama|War,100 greatest movies 70mm 70mm film abandoned building afi 100 afi 100 cheers afi 100 thrills afi...,4.134257,15772,4.132827,0.911598,0.837482,0.393441,0.272597,0.380117,obscure,"Coincide con tu perfil por sus géneros Drama, Adventure, War y por temas como shot, death, story..."
7,Knives Out (2019),2019,Comedy|Crime|Drama|Mystery|Thriller,16 year old boy 21th century accomplice activist adulterous husband adultery aerial camera shot ...,4.042505,10740,4.040576,0.884580,0.804192,0.413747,0.326613,0.379423,obscure,"Coincide con tu perfil por sus géneros Drama, Comedy, Thriller y por temas como ending, shot, de..."
8,Fight Club (1999),1999,Action|Crime|Drama|Thriller,20th century fox 20th century studios 5 stars abandoned house absolute favorite acid acting acti...,4.228780,77332,4.228463,0.939608,0.975227,0.427883,0.342162,0.378696,obscure,"Coincide con tu perfil por sus géneros Drama, Thriller, Crime y por temas como ending, nudity, s..."
9,Roma (2018),2018,Drama,abandonment absorbing absurdism academic acting adultery airplane alfonso cuaron ambiance artsy ...,3.778936,1918,3.770950,0.805614,0.654972,0.391160,0.266051,0.377112,obscure,"Coincide con tu perfil por sus géneros Drama y por temas como nudity, shot, death. tiene una val..."


,title,year,genres,tags_combined_en,rating_mean,rating_count,weighted_rating,rating_score,popularity_score,user_similarity_score,negative_similarity_score,final_score,recommendation_mode,explanation
0,"Shawshank Redemption, The (1994)",1994,Crime|Drama,100 greatest movies 20th century 20th century literature on screen 5 stars abuse abuse of power ...,4.404614,102929,4.404342,0.991118,1.000000,0.445944,0.314574,0.578082,quality,"Coincide con tu perfil por sus géneros Drama, Crime y por temas como ending, nudity, shot. tiene..."
1,Schindler's List (1993),1993,Drama|War,afi 100 afi 100 cheers afi 100 heroes villains afi 100 movie quotes afi100 altruism amazing phot...,4.236990,73849,4.236657,0.942007,0.971234,0.397414,0.243299,0.549296,quality,"Coincide con tu perfil por sus géneros Drama, War y por temas como nudity, story, music. tiene u..."
2,Fight Club (1999),1999,Action|Crime|Drama|Thriller,20th century fox 20th century studios 5 stars abandoned house absolute favorite acid acting acti...,4.228780,77332,4.228463,0.939608,0.975227,0.427883,0.342162,0.546214,quality,"Coincide con tu perfil por sus géneros Drama, Thriller, Crime y por temas como ending, nudity, s..."
3,One Flew Over the Cuckoo's Nest (1975),1975,Drama,acting adapted from book afi 100 afi 100 cheers afi 100 heroes villains allegory angry anjelica ...,4.204229,44592,4.203692,0.932353,0.927527,0.402046,0.246861,0.542865,quality,"Coincide con tu perfil por sus géneros Drama y por temas como ending, nudity, acting. tiene una ..."
4,"Usual Suspects, The (1995)",1995,Crime|Mystery|Thriller,acting action ambiguous ending anti-hero atmosphere bad acting bd-video benicio del toro boring ...,4.265070,67750,4.264698,0.950220,0.963766,0.349693,0.268495,0.528556,quality,"Coincide con tu perfil por sus géneros Thriller, Crime, Mystery y por temas como ending, story, ..."
5,Terminator 2: Judgment Day (1991),1991,Action|Sci-Fi,10 year old 20th century 21st century 70mm acquaintance action action girl action hero action he...,3.962045,68383,3.961765,0.861499,0.964571,0.469830,0.387507,0.527788,quality,"Coincide con tu perfil por sus géneros Sci-Fi, Action y por temas como ending, nudity, shot. tie..."
6,Amadeus (1984),1984,Drama,18th century 70mm afi 100 ambition anamorphic blow-up austria bad acting based on a play beautif...,4.072641,24986,4.071787,0.893721,0.877342,0.382928,0.220587,0.520620,quality,"Coincide con tu perfil por sus géneros Drama y por temas como nudity, story, music. tiene una va..."
7,Saving Private Ryan (1998),1998,Action|Drama|War,20th century act of god action adam goldberg afi 100 cheers afi 100 thrills airplane almost favo...,4.048742,58368,4.048385,0.886867,0.950851,0.403171,0.313384,0.519750,quality,"Coincide con tu perfil por sus géneros Drama, Action, War y por temas como shot, death, story. t..."
8,Lawrence of Arabia (1962),1962,Adventure|Drama|War,100 greatest movies 70mm 70mm film abandoned building afi 100 afi 100 cheers afi 100 thrills afi...,4.134257,15772,4.132827,0.911598,0.837482,0.393441,0.272597,0.519294,quality,"Coincide con tu perfil por sus géneros Drama, Adventure, War y por temas como shot, death, story..."
9,Back to the Future (1985),1985,Adventure|Comedy|Sci-Fi,1980s film 20th century 55 movies every kid should see--entertainment weekly 555 phone number 70...,3.969149,61879,3.968838,0.863570,0.955912,0.427165,0.335499,0.518382,quality,"Coincide con tu perfil por sus géneros Comedy, Sci-Fi, Adventure y por temas como ending, shot, ..."


,mode,title,year,final_score
0,balanced,"Shawshank Redemption, The (1994)",1994,0.348535
1,balanced,Terminator 2: Judgment Day (1991),1991,0.339591
2,balanced,"Terminator, The (1984)",1984,0.327930
3,balanced,Fight Club (1999),1999,0.325759
4,balanced,One Flew Over the Cuckoo's Nest (1975),1975,0.325198
5,balanced,Schindler's List (1993),1993,0.324315
6,balanced,"Graduate, The (1967)",1967,0.324122
7,balanced,Back to the Future (1985),1985,0.320120
8,balanced,There Will Be Blood (2007),2007,0.315507
9,balanced,Children of Men (2006),2006,0.313031


## 12. Modos y resumen

Los modos de recomendacion se interpretan asi:
- `balanced`: prioriza el encaje con el perfil del usuario y solo usa popularidad como apoyo.
- `popular`: da mas peso a peliculas conocidas y con mucha presencia.
- `quality`: prioriza la valoracion ponderada.
- `under_the_radar` y `obscure`: ayudan a descubrir peliculas menos evidentes, manteniendo relevancia para el perfil.

Con este notebook ya no solo extraemos datos de Trakt: tambien los alineamos con MovieLens y los dejamos listos para alimentar un perfil de usuario explicable.

La ventaja es que todo queda trazable: autenticacion, descarga, mapeo de IDs, construccion del perfil y generacion de recomendaciones.